# Enterprise Sales & Commercial Analytics Platform

### SQL-First Enterprise Data Warehouse & Business Intelligence Project

---

## Project Overview

This project implements an **Enterprise Sales & Commercial Intelligence Platform** using SQL as the primary analytical engine.

The platform integrates data from CRM, ERP, Finance, Marketing, Inventory, and Customer Success into a centralized data warehouse to support standardized reporting, KPI management, and executive decision-making.

It demonstrates how enterprise organizations design analytical systems to consolidate operational data, improve reporting consistency, and provide a single source of truth for commercial performance.

---

## Business Problem

Business data is distributed across multiple operational systems, resulting in:

- Inconsistent KPI definitions
- Conflicting revenue reports across departments
- Manual reporting processes
- Limited visibility into commercial performance
- No centralized analytics platform for decision-making

---

## Project Objectives

- Design a Star Schema data warehouse
- Build a SQL-first ETL pipeline
- Standardize business KPIs using governed SQL views
- Perform advanced SQL analytics
- Develop executive dashboards
- Generate business insights and strategic recommendations

---

## Architecture Overview

```text
                  Enterprise Source Systems
 ┌──────────┬──────────┬──────────┬──────────┬──────────┬─────────────────┐
 │   CRM    │   ERP    │ Finance  │ Marketing│ Inventory│ Customer Success│
 └──────────┴──────────┴──────────┴──────────┴──────────┴─────────────────┘
                                │
                                ▼
                     Enterprise Data Generation
                                │
                                ▼
                        Staging & ETL Pipeline
                                │
                                ▼
                 Star Schema Data Warehouse (SQLite)
                                │
                                ▼
                Governed SQL Views & Business Logic
                                │
                                ▼
              KPI Reporting • Advanced SQL Analytics
                                │
                                ▼
        Python Analytics • Forecasting • Executive Dashboards
                                │
                                ▼
              Business Insights & Strategic Recommendations
```

---

## Technology Stack

| Component | Technology |
|-----------|------------|
| Database | SQLite |
| Primary Language | SQL |
| Supporting Language | Python |
| Visualization | Plotly, Matplotlib |
| Development Environment | Visual Studio Code |

---

## Project Workflow

1. Generate enterprise source data
2. Build the Star Schema data warehouse
3. Load data through the ETL pipeline
4. Validate data quality
5. Create governed SQL views and business logic
6. Perform advanced SQL analytics
7. Develop executive dashboards
8. Generate business insights and recommendations

In [ ]:
!pip install -q pandas numpy sqlalchemy plotly matplotlib scikit-learn scipy statsmodels openpyxl faker

import os
import sqlite3
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

np.random.seed(42)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

DB_PATH = "enterprise_sales_ci.db"

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

print(f"Environment initialized. Database: {DB_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.3 MB/s eta 0:00:00
Environment initialized. Database: enterprise_sales_ci.db


# 1. Source Data Generation

## Enterprise Data Landscape

The platform integrates data typically managed across six enterprise business systems:

- Customer Relationship Management (CRM)
- Enterprise Resource Planning (ERP)
- Finance
- Marketing
- Inventory
- Customer Success

For demonstration purposes, this project generates a realistic enterprise dataset that mirrors the structure and relationships commonly found in commercial analytics environments.

---

## Synthetic Data Strategy

The generated dataset includes realistic business entities, transactional data, and operational inconsistencies that are commonly encountered during enterprise data integration.

This approach enables the complete analytics pipeline—including ETL, data quality validation, KPI calculations, and reporting—to be demonstrated without relying on external data sources.


In [ ]:
from faker import Faker

fake = Faker()
Faker.seed(42)

N_CUSTOMERS = 1200
N_REPS = 45
N_PRODUCTS = 180
N_SUPPLIERS = 30
N_REGIONS = 12
N_CAMPAIGNS = 60
N_OPPS = 4200
N_ORDERS = 9000

DATE_START = datetime(2022, 1, 1)
DATE_END = datetime(2024, 12, 31)
DIM_DATE_END = DATE_END + timedelta(days=45)

# DimDate
dim_date = pd.DataFrame({"date": pd.date_range(DATE_START, DIM_DATE_END, freq="D")})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_of_week"] = dim_date["date"].dt.dayofweek
dim_date["day_name"] = dim_date["date"].dt.strftime("%a")
dim_date["is_weekend"] = dim_date["day_of_week"].isin([5, 6]).astype(int)
dim_date["fiscal_year"] = dim_date["year"]

dim_date = dim_date[
    [
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_name",
        "is_weekend",
        "fiscal_year",
    ]
]

# DimRegion
regions_raw = [
    ("North America", "USA", "New York"),
    ("North America", "USA", "Chicago"),
    ("North America", "Canada", "Toronto"),
    ("Europe", "UK", "London"),
    ("Europe", "Germany", "Berlin"),
    ("Europe", "France", "Paris"),
    ("Asia Pacific", "Australia", "Sydney"),
    ("Asia Pacific", "Japan", "Tokyo"),
    ("Asia Pacific", "Singapore", "Singapore"),
    ("Latin America", "Brazil", "Sao Paulo"),
    ("Latin America", "Mexico", "Mexico City"),
    ("Middle East", "UAE", "Dubai"),
]

dim_region = pd.DataFrame(
    regions_raw,
    columns=["continent", "country", "city"]
)

dim_region.insert(0, "region_key", range(1, len(dim_region) + 1))
dim_region["region_name"] = dim_region["city"] + " Territory"

# DimCurrency
dim_currency = pd.DataFrame({
    "currency_key": [1, 2, 3, 4],
    "currency_code": ["USD", "EUR", "GBP", "AUD"],
    "usd_exchange_rate": [1.0, 1.08, 1.27, 0.66],
})

# DimChannel
dim_channel = pd.DataFrame({
    "channel_key": [1, 2, 3, 4],
    "channel_name": [
        "Direct Sales",
        "Partner/Reseller",
        "E-Commerce",
        "Inside Sales",
    ],
})

# DimIndustry
industries = [
    "Technology",
    "Manufacturing",
    "Retail",
    "Healthcare",
    "Financial Services",
    "Energy",
    "Telecommunications",
    "Public Sector",
]

dim_industry = pd.DataFrame({
    "industry_key": range(1, len(industries) + 1),
    "industry_name": industries,
})

# DimSalesRep
dim_salesrep = pd.DataFrame({
    "rep_key": range(1, N_REPS + 1),
    "rep_name": [fake.name() for _ in range(N_REPS)],
    "region_key": np.random.choice(dim_region["region_key"], N_REPS),
    "hire_date": [
        fake.date_between(datetime(2016, 1, 1), datetime(2023, 1, 1))
        for _ in range(N_REPS)
    ],
    "annual_quota_usd": np.random.choice(
        [600000, 800000, 1000000, 1250000, 1500000],
        N_REPS,
    ),
    "seniority": np.random.choice(
        ["Junior", "Mid", "Senior", "Principal"],
        N_REPS,
        p=[0.25, 0.35, 0.30, 0.10],
    ),
})

# DimSupplier
dim_supplier = pd.DataFrame({
    "supplier_key": range(1, N_SUPPLIERS + 1),
    "supplier_name": [
        fake.company() + " Supply Co."
        for _ in range(N_SUPPLIERS)
    ],
    "country": np.random.choice(
        dim_region["country"].unique(),
        N_SUPPLIERS,
    ),
    "avg_lead_time_days": np.random.randint(3, 45, N_SUPPLIERS),
    "on_time_delivery_rate": np.round(
        np.random.uniform(0.72, 0.99, N_SUPPLIERS),
        3,
    ),
})

# DimProduct
categories = {
    "Enterprise Software": (250, 7500),
    "Hardware": (75, 2000),
    "Cloud Services": (25, 1000),
    "Professional Services": (150, 5000),
    "Support & Maintenance": (50, 750),
    "Analytics Add-ons": (75, 1500),
}
cat_names = list(categories.keys())

prod_cats = np.random.choice(
    cat_names,
    N_PRODUCTS,
    p=[0.22, 0.18, 0.22, 0.14, 0.14, 0.10],
)

unit_costs = []
list_prices = []

for c in prod_cats:
    lo, hi = categories[c]
    cost = np.random.uniform(lo, hi)
    price = cost * np.random.uniform(1.35, 2.6)

    unit_costs.append(round(cost, 2))
    list_prices.append(round(price, 2))

dim_product = pd.DataFrame({
    "product_key": range(1, N_PRODUCTS + 1),
    "product_name": [
        f"{c.split()[0]}-{i:04d}"
        for i, c in zip(range(1, N_PRODUCTS + 1), prod_cats)
    ],
    "category": prod_cats,
    "supplier_key": np.random.choice(
        dim_supplier["supplier_key"],
        N_PRODUCTS,
    ),
    "unit_cost": unit_costs,
    "list_price": list_prices,
    "is_active": np.random.choice(
        [1, 0],
        N_PRODUCTS,
        p=[0.9, 0.1],
    ),
})

# DimCustomer
segments = ["Enterprise", "Mid-Market", "SMB"]

dim_customer = pd.DataFrame({
    "customer_key": range(1, N_CUSTOMERS + 1),
    "customer_name": [fake.company() for _ in range(N_CUSTOMERS)],
    "industry_key": np.random.choice(
        dim_industry["industry_key"],
        N_CUSTOMERS,
    ),
    "region_key": np.random.choice(
        dim_region["region_key"],
        N_CUSTOMERS,
    ),
    "segment": np.random.choice(
        segments,
        N_CUSTOMERS,
        p=[0.15, 0.35, 0.50],
    ),
    "signup_date": [
        fake.date_between(
            DATE_START,
            DATE_END - timedelta(days=30),
        )
        for _ in range(N_CUSTOMERS)
    ],
    "email": [
        fake.company_email()
        for _ in range(N_CUSTOMERS)
    ],
})

# Simulate common enterprise data quality issues.
dim_customer.loc[
    dim_customer.sample(frac=0.03, random_state=1).index,
    "email",
] = None

dupe_rows = dim_customer.sample(frac=0.01, random_state=2).copy()
dim_customer = pd.concat([dim_customer, dupe_rows], ignore_index=False)

# DimPromotion
dim_promotion = pd.DataFrame({
    "promotion_key": range(1, 21),
    "promotion_name": [f"Promo-{i:02d}" for i in range(1, 21)],
    "discount_pct": np.round(
        np.random.uniform(0.02, 0.30, 20),
        2,
    ),
    "start_date": [
        fake.date_between(
            DATE_START,
            DATE_END - timedelta(days=60),
        )
        for _ in range(20)
    ],
})

dim_promotion["end_date"] = (
    pd.to_datetime(dim_promotion["start_date"])
    + pd.to_timedelta(np.random.randint(14, 90, 20), unit="D")
)

dimension_summary = pd.DataFrame({
    "Dimension": [
        "DimCustomer",
        "DimProduct",
        "DimSalesRep",
        "DimRegion",
        "DimSupplier",
        "DimDate",
    ],
    "Rows": [
        len(dim_customer),
        len(dim_product),
        len(dim_salesrep),
        len(dim_region),
        len(dim_supplier),
        len(dim_date),
    ],
})

print("Dimension tables generated successfully.")

display(dimension_summary.style.hide(axis="index"))

Dimension tables generated successfully.


Dimension,Rows
DimCustomer,1212
DimProduct,180
DimSalesRep,45
DimRegion,12
DimSupplier,30
DimDate,1141


### Dimension Summary

The following dimension tables provide the descriptive attributes used throughout the enterprise data warehouse. These tables establish the business entities that support analytical reporting across Sales, Finance, Marketing, Inventory, and Customer Success.

The transactional fact tables are generated in the next section and will reference these dimensions through foreign key relationships.

# 2. Transactional / Event Data (Fact Sources)

## Fact Table Overview

The following fact tables capture the operational transactions generated across the organization's core business systems. These tables record business events such as sales transactions, customer opportunities, marketing campaigns, inventory movements, product returns, demand forecasts, and customer interactions.

Together with the dimension tables created in the previous section, these datasets form the analytical foundation of the enterprise data warehouse and support KPI reporting, trend analysis, forecasting, customer analytics, and executive dashboards.

In [ ]:
active_customers = dim_customer["customer_key"].unique()
active_products   = dim_product.loc[dim_product["is_active"] == 1, "product_key"].values
all_dates         = dim_date["date"].values

def random_date_between(start, end, n):
    start_u = pd.Timestamp(start).value // 10**9
    end_u   = pd.Timestamp(end).value // 10**9
    return pd.to_datetime(np.random.randint(start_u, end_u, n), unit="s").normalize()

# FactOpportunities
opp_created = random_date_between(DATE_START, DATE_END - timedelta(days=1), N_OPPS)
stage_probs   = np.random.rand(N_OPPS)
stage         = np.where(stage_probs < 0.55, "Closed Won",
                 np.where(stage_probs < 0.80, "Closed Lost",
                 np.where(stage_probs < 0.90, "Negotiation",
                 np.where(stage_probs < 0.96, "Proposal", "Qualification"))))
cycle_days    = np.random.randint(7, 180, N_OPPS)
fact_opportunities = pd.DataFrame({
    "opportunity_key": range(1, N_OPPS + 1),
    "customer_key": np.random.choice(active_customers, N_OPPS),
    "rep_key": np.random.choice(dim_salesrep["rep_key"], N_OPPS),
    "channel_key": np.random.choice(dim_channel["channel_key"], N_OPPS),
    "created_date_key": opp_created.strftime("%Y%m%d").astype(int),
    "expected_close_date_key": (opp_created + pd.to_timedelta(cycle_days, unit="D")).strftime("%Y%m%d").astype(int),
    "stage": stage,
    "deal_amount_usd": np.round(
    np.random.lognormal(mean=11.6, sigma=0.9, size=N_OPPS),
    2
).clip(1500, 500000),
    "sales_cycle_days": cycle_days,
})
# Simulate referential integrity exceptions.
bad_idx = fact_opportunities.sample(frac=0.005, random_state=3).index
fact_opportunities.loc[bad_idx, "rep_key"] = 9999

# FactOrders & FactSales
order_dates  = random_date_between(DATE_START, DATE_END, N_ORDERS)
won_opps     = fact_opportunities[fact_opportunities["stage"] == "Closed Won"]
order_customers = np.random.choice(
    won_opps["customer_key"].values if len(won_opps) > N_ORDERS else active_customers, N_ORDERS)

fact_orders = pd.DataFrame({
    "order_key": range(1, N_ORDERS + 1),
    "customer_key": order_customers,
    "rep_key": np.random.choice(dim_salesrep["rep_key"], N_ORDERS),
    "channel_key": np.random.choice(dim_channel["channel_key"], N_ORDERS),
    "region_key": np.random.choice(dim_region["region_key"], N_ORDERS),
    "currency_key": np.random.choice(dim_currency["currency_key"], N_ORDERS, p=[0.6, 0.2, 0.15, 0.05]),
    "order_date_key": order_dates.strftime("%Y%m%d").astype(int),
    "promotion_key": np.random.choice(
        list(dim_promotion["promotion_key"]) + [None] * 3, N_ORDERS),
})

line_rows = []
line_id = 1
for _, o in fact_orders.iterrows():
    n_lines = np.random.randint(1, 5)
    for _ in range(n_lines):
        pk = np.random.choice(active_products)
        prod = dim_product.loc[dim_product["product_key"] == pk].iloc[0]
        qty = np.random.randint(1, 25)
        discount_pct = np.random.choice([0, 0, 0, 0.05, 0.10, 0.15, 0.20], p=[0.35,0.15,0.1,0.15,0.1,0.1,0.05])
        gross = round(prod["list_price"] * qty, 2)
        net   = round(gross * (1 - discount_pct), 2)
        cost  = round(prod["unit_cost"] * qty, 2)
        line_rows.append((line_id, o["order_key"], pk, o["customer_key"], o["order_date_key"],
                           qty, prod["list_price"], discount_pct, gross, net, cost))
        line_id += 1

fact_sales = pd.DataFrame(line_rows, columns=[
    "sales_line_key", "order_key", "product_key", "customer_key", "date_key",
    "quantity", "unit_list_price", "discount_pct", "gross_revenue", "net_revenue", "total_cost"])

# Assign realistic annual quotas based on generated sales.

rep_revenue = (
    fact_orders
    .merge(
        fact_sales.groupby("order_key", as_index=False)["net_revenue"].sum(),
        on="order_key",
    )
    .groupby("rep_key", as_index=False)["net_revenue"].sum()
)

target_attainment = np.random.normal(
    loc=1.0,
    scale=0.18,
    size=len(rep_revenue),
)

target_attainment = np.clip(target_attainment, 0.70, 1.40)

rep_revenue["annual_quota_usd"] = (
    rep_revenue["net_revenue"] / target_attainment
).round(0)

dim_salesrep = (
    dim_salesrep.drop(columns=["annual_quota_usd"])
    .merge(
        rep_revenue[["rep_key", "annual_quota_usd"]],
        on="rep_key",
        how="left",
    )
)

median_quota = rep_revenue["annual_quota_usd"].median()

dim_salesrep["annual_quota_usd"] = (
    dim_salesrep["annual_quota_usd"]
    .fillna(median_quota)
)

# FactReturns
n_returns = int(len(fact_sales) * 0.045)
returned_lines = fact_sales.sample(n_returns, random_state=4)
fact_returns = pd.DataFrame({
    "return_key": range(1, n_returns + 1),
    "sales_line_key": returned_lines["sales_line_key"].values,
    "product_key": returned_lines["product_key"].values,
    "customer_key": returned_lines["customer_key"].values,
    "return_date_key": (pd.to_datetime(returned_lines["date_key"].astype(str)) +
                         pd.to_timedelta(np.random.randint(1, 30, n_returns), unit="D")
                         ).dt.strftime("%Y%m%d").astype(int),
    "returned_qty": np.minimum(returned_lines["quantity"].values,
                                np.random.randint(1, 4, n_returns)),
    "refund_amount": np.round(returned_lines["net_revenue"].values *
                               np.random.uniform(0.5, 1.0, n_returns), 2),
    "return_reason": np.random.choice(
        ["Defective", "Wrong Item", "No Longer Needed", "Better Price Found", "Late Delivery"],
        n_returns),
})

# FactMarketing
campaign_dates = random_date_between(DATE_START, DATE_END - timedelta(days=30), N_CAMPAIGNS)
channels_mkt = ["Paid Search", "Paid Social", "Email", "Events/Webinars", "Content/SEO", "Partner Co-Marketing"]
spend = np.round(np.random.lognormal(9.5, 0.6, N_CAMPAIGNS), 2)
leads = np.random.randint(50, 3000, N_CAMPAIGNS)
mqls  = (leads * np.random.uniform(0.2, 0.5, N_CAMPAIGNS)).astype(int)
sqls  = (mqls * np.random.uniform(0.25, 0.55, N_CAMPAIGNS)).astype(int)
opps_generated = (sqls * np.random.uniform(0.3, 0.6, N_CAMPAIGNS)).astype(int)
won_deals = (opps_generated * np.random.uniform(0.15, 0.4, N_CAMPAIGNS)).astype(int)
revenue_influenced = won_deals * np.random.uniform(4000, 25000, N_CAMPAIGNS)

fact_marketing = pd.DataFrame({
    "campaign_key": range(1, N_CAMPAIGNS + 1),
    "campaign_name": [f"CMP-{i:03d}" for i in range(1, N_CAMPAIGNS + 1)],
    "channel": np.random.choice(channels_mkt, N_CAMPAIGNS),
    "start_date_key": campaign_dates.strftime("%Y%m%d").astype(int),
    "spend_usd": spend,
    "leads_generated": leads,
    "mqls": mqls,
    "sqls": sqls,
    "opportunities_generated": opps_generated,
    "deals_won": won_deals,
    "revenue_influenced_usd": np.round(revenue_influenced, 2),
})

# FactInventory
inv_rows = []
snapshot_months = pd.date_range(DATE_START, DATE_END, freq="MS")
for pk in dim_product["product_key"]:
    supplier_key = dim_product.loc[dim_product["product_key"] == pk, "supplier_key"].iloc[0]
    on_hand = np.random.randint(0, 500)
    for d in snapshot_months:
        demand = np.random.poisson(30)
        on_hand = max(0, on_hand + np.random.randint(-40, 60) - demand)
        inv_rows.append((pk, supplier_key, int(d.strftime("%Y%m%d")), on_hand, demand,
                          1 if on_hand == 0 else 0))
fact_inventory = pd.DataFrame(inv_rows, columns=[
    "product_key", "supplier_key", "snapshot_date_key", "units_on_hand", "units_demanded", "stockout_flag"])
fact_inventory.insert(0, "inventory_key", range(1, len(fact_inventory) + 1))

# FactForecast
fc_rows = []
for pk in dim_product["product_key"]:
    for d in snapshot_months:
        actual = fact_inventory[(fact_inventory.product_key == pk) &
                                 (fact_inventory.snapshot_date_key == int(d.strftime("%Y%m%d")))]["units_demanded"]
        actual_val = int(actual.iloc[0]) if len(actual) else np.random.poisson(30)
        forecast_val = max(0, int(actual_val * np.random.uniform(0.75, 1.25)))
        fc_rows.append((pk, int(d.strftime("%Y%m%d")), forecast_val, actual_val))
fact_forecast = pd.DataFrame(fc_rows, columns=["product_key", "period_date_key", "forecast_units", "actual_units"])
fact_forecast.insert(0, "forecast_key", range(1, len(fact_forecast) + 1))

# FactCustomerActivity
n_activity = N_CUSTOMERS * 6
fact_customer_activity = pd.DataFrame({
    "activity_key": range(1, n_activity + 1),
    "customer_key": np.random.choice(active_customers, n_activity),
    "activity_date_key": random_date_between(DATE_START, DATE_END, n_activity).strftime("%Y%m%d").astype(int),
    "activity_type": np.random.choice(
        ["Support Ticket", "QBR", "Product Usage Login", "NPS Survey", "Renewal Touchpoint", "Escalation"],
        n_activity, p=[0.30, 0.10, 0.35, 0.10, 0.10, 0.05]),
    "health_score": np.random.randint(1, 101, n_activity),
    "csat_score": np.random.choice([1,2,3,4,5, None], n_activity, p=[0.03,0.05,0.12,0.35,0.40,0.05]),
})

fact_summary = pd.DataFrame({
    "Fact Table": [
        "FactOpportunities",
        "FactOrders",
        "FactSales",
        "FactReturns",
        "FactMarketing",
        "FactInventory",
        "FactForecast",
        "FactCustomerActivity",
    ],
    "Rows": [
        len(fact_opportunities),
        len(fact_orders),
        len(fact_sales),
        len(fact_returns),
        len(fact_marketing),
        len(fact_inventory),
        len(fact_forecast),
        len(fact_customer_activity),
    ],
})

print("Fact tables generated successfully.")

display(fact_summary.style.hide(axis="index"))


Fact tables generated successfully.


Fact Table,Rows
FactOpportunities,4200
FactOrders,9000
FactSales,22645
FactReturns,1019
FactMarketing,60
FactInventory,6480
FactForecast,6480
FactCustomerActivity,7200


## Enterprise Dataset Summary

The enterprise dataset now includes both master data (dimension tables) and transactional data (fact tables), providing a complete analytical foundation for the data warehouse.

The dimension tables define the core business entities, while the fact tables capture operational events across CRM, ERP, Finance, Marketing, Inventory, and Customer Success. Together, they support the KPI calculations, SQL analytics, forecasting models, and executive dashboards developed throughout the remainder of this project.

# 3. Star Schema Data Warehouse (SQLite)

The warehouse consolidates data from multiple business systems into a single analytical model used throughout this project. It separates business entities from transactional data, making KPI calculations, reporting, and analytical queries easier to build and maintain.

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

DDL_STATEMENTS = [
# Dimension Tables
"""CREATE TABLE DimDate (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL,
    year INTEGER NOT NULL,
    quarter INTEGER NOT NULL CHECK (quarter BETWEEN 1 AND 4),
    month INTEGER NOT NULL CHECK (month BETWEEN 1 AND 12),
    month_name TEXT NOT NULL,
    day INTEGER NOT NULL,
    day_of_week INTEGER NOT NULL,
    day_name TEXT NOT NULL,
    is_weekend INTEGER NOT NULL CHECK (is_weekend IN (0,1)),
    fiscal_year INTEGER NOT NULL
);""",
"""CREATE TABLE DimRegion (
    region_key INTEGER PRIMARY KEY,
    region_name TEXT NOT NULL,
    continent TEXT NOT NULL,
    country TEXT NOT NULL,
    city TEXT NOT NULL
);""",
"""CREATE TABLE DimCurrency (
    currency_key INTEGER PRIMARY KEY,
    currency_code TEXT NOT NULL UNIQUE,
    usd_exchange_rate REAL NOT NULL CHECK (usd_exchange_rate > 0)
);""",
"""CREATE TABLE DimChannel (
    channel_key INTEGER PRIMARY KEY,
    channel_name TEXT NOT NULL UNIQUE
);""",
"""CREATE TABLE DimIndustry (
    industry_key INTEGER PRIMARY KEY,
    industry_name TEXT NOT NULL UNIQUE
);""",
"""CREATE TABLE DimSupplier (
    supplier_key INTEGER PRIMARY KEY,
    supplier_name TEXT NOT NULL,
    country TEXT,
    avg_lead_time_days INTEGER CHECK (avg_lead_time_days >= 0),
    on_time_delivery_rate REAL CHECK (on_time_delivery_rate BETWEEN 0 AND 1)
);""",
"""CREATE TABLE DimSalesRep (
    rep_key INTEGER PRIMARY KEY,
    rep_name TEXT NOT NULL,
    region_key INTEGER REFERENCES DimRegion(region_key),
    hire_date TEXT,
    annual_quota_usd REAL CHECK (annual_quota_usd > 0),
    seniority TEXT
);""",
"""CREATE TABLE DimProduct (
    product_key INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    supplier_key INTEGER REFERENCES DimSupplier(supplier_key),
    unit_cost REAL NOT NULL CHECK (unit_cost >= 0),
    list_price REAL NOT NULL CHECK (list_price >= 0),
    is_active INTEGER NOT NULL CHECK (is_active IN (0,1))
);""",
"""CREATE TABLE DimCustomer (
    customer_key INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    industry_key INTEGER REFERENCES DimIndustry(industry_key),
    region_key INTEGER REFERENCES DimRegion(region_key),
    segment TEXT NOT NULL CHECK (segment IN ('Enterprise','Mid-Market','SMB')),
    signup_date TEXT,
    email TEXT
);""",
"""CREATE TABLE DimPromotion (
    promotion_key INTEGER PRIMARY KEY,
    promotion_name TEXT NOT NULL,
    discount_pct REAL CHECK (discount_pct BETWEEN 0 AND 1),
    start_date TEXT,
    end_date TEXT
);""",
# Fact Tables
"""CREATE TABLE FactOpportunities (
    opportunity_key INTEGER PRIMARY KEY,
    customer_key INTEGER REFERENCES DimCustomer(customer_key),
    rep_key INTEGER REFERENCES DimSalesRep(rep_key),
    channel_key INTEGER REFERENCES DimChannel(channel_key),
    created_date_key INTEGER REFERENCES DimDate(date_key),
    expected_close_date_key INTEGER,
    stage TEXT NOT NULL CHECK (stage IN ('Qualification','Proposal','Negotiation','Closed Won','Closed Lost')),
    deal_amount_usd REAL CHECK (deal_amount_usd >= 0),
    sales_cycle_days INTEGER
);""",
"""CREATE TABLE FactOrders (
    order_key INTEGER PRIMARY KEY,
    customer_key INTEGER REFERENCES DimCustomer(customer_key),
    rep_key INTEGER REFERENCES DimSalesRep(rep_key),
    channel_key INTEGER REFERENCES DimChannel(channel_key),
    region_key INTEGER REFERENCES DimRegion(region_key),
    currency_key INTEGER REFERENCES DimCurrency(currency_key),
    order_date_key INTEGER REFERENCES DimDate(date_key),
    promotion_key INTEGER REFERENCES DimPromotion(promotion_key)
);""",
"""CREATE TABLE FactSales (
    sales_line_key INTEGER PRIMARY KEY,
    order_key INTEGER REFERENCES FactOrders(order_key),
    product_key INTEGER REFERENCES DimProduct(product_key),
    customer_key INTEGER REFERENCES DimCustomer(customer_key),
    date_key INTEGER REFERENCES DimDate(date_key),
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_list_price REAL NOT NULL,
    discount_pct REAL NOT NULL CHECK (discount_pct BETWEEN 0 AND 1),
    gross_revenue REAL NOT NULL,
    net_revenue REAL NOT NULL,
    total_cost REAL NOT NULL
);""",
"""CREATE TABLE FactReturns (
    return_key INTEGER PRIMARY KEY,
    sales_line_key INTEGER REFERENCES FactSales(sales_line_key),
    product_key INTEGER REFERENCES DimProduct(product_key),
    customer_key INTEGER REFERENCES DimCustomer(customer_key),
    return_date_key INTEGER REFERENCES DimDate(date_key),
    returned_qty INTEGER CHECK (returned_qty >= 0),
    refund_amount REAL CHECK (refund_amount >= 0),
    return_reason TEXT
);""",
"""CREATE TABLE FactMarketing (
    campaign_key INTEGER PRIMARY KEY,
    campaign_name TEXT NOT NULL,
    channel TEXT NOT NULL,
    start_date_key INTEGER REFERENCES DimDate(date_key),
    spend_usd REAL CHECK (spend_usd >= 0),
    leads_generated INTEGER,
    mqls INTEGER,
    sqls INTEGER,
    opportunities_generated INTEGER,
    deals_won INTEGER,
    revenue_influenced_usd REAL
);""",
"""CREATE TABLE FactInventory (
    inventory_key INTEGER PRIMARY KEY,
    product_key INTEGER REFERENCES DimProduct(product_key),
    supplier_key INTEGER REFERENCES DimSupplier(supplier_key),
    snapshot_date_key INTEGER REFERENCES DimDate(date_key),
    units_on_hand INTEGER CHECK (units_on_hand >= 0),
    units_demanded INTEGER CHECK (units_demanded >= 0),
    stockout_flag INTEGER CHECK (stockout_flag IN (0,1))
);""",
"""CREATE TABLE FactForecast (
    forecast_key INTEGER PRIMARY KEY,
    product_key INTEGER REFERENCES DimProduct(product_key),
    period_date_key INTEGER REFERENCES DimDate(date_key),
    forecast_units INTEGER,
    actual_units INTEGER
);""",
"""CREATE TABLE FactCustomerActivity (
    activity_key INTEGER PRIMARY KEY,
    customer_key INTEGER REFERENCES DimCustomer(customer_key),
    activity_date_key INTEGER REFERENCES DimDate(date_key),
    activity_type TEXT NOT NULL,
    health_score INTEGER CHECK (health_score BETWEEN 1 AND 100),
    csat_score INTEGER
);""",
]

for stmt in DDL_STATEMENTS:
    cur.execute(stmt)

INDEX_STATEMENTS = [
    "CREATE INDEX idx_sales_date ON FactSales(date_key);",
    "CREATE INDEX idx_sales_product ON FactSales(product_key);",
    "CREATE INDEX idx_sales_customer ON FactSales(customer_key);",
    "CREATE INDEX idx_orders_date ON FactOrders(order_date_key);",
    "CREATE INDEX idx_orders_rep ON FactOrders(rep_key);",
    "CREATE INDEX idx_opps_stage ON FactOpportunities(stage);",
    "CREATE INDEX idx_opps_rep ON FactOpportunities(rep_key);",
    "CREATE INDEX idx_opps_created ON FactOpportunities(created_date_key);",
    "CREATE INDEX idx_inventory_product ON FactInventory(product_key);",
    "CREATE INDEX idx_inventory_snapshot ON FactInventory(snapshot_date_key);",
    "CREATE INDEX idx_activity_customer ON FactCustomerActivity(customer_key);",
    "CREATE INDEX idx_returns_product ON FactReturns(product_key);",
    "CREATE INDEX idx_customer_segment ON DimCustomer(segment);",
]
for stmt in INDEX_STATEMENTS:
    cur.execute(stmt)

conn.commit()
warehouse_summary = pd.DataFrame({
    "Component": ["Tables", "Indexes"],
    "Count": [len(DDL_STATEMENTS), len(INDEX_STATEMENTS)],
})

print("Data warehouse created successfully.")

display(warehouse_summary.style.hide(axis="index"))

Data warehouse created successfully.


Component,Count
Tables,18
Indexes,13


# 4. ETL — Staging, Data-Quality Rules, Incremental-Style Load

Operational data is first loaded into staging tables before being validated, cleaned, and loaded into the analytical warehouse.

This process applies data quality checks, resolves common data issues, and ensures the warehouse contains consistent, reliable data for reporting and analysis.

In [ ]:
dim_date_stg = dim_date.rename(columns={"date": "full_date"}).copy()
dim_date_stg["full_date"] = dim_date_stg["full_date"].astype(str)

staging_map = {
    "stg_DimDate": dim_date_stg,
    "stg_DimRegion": dim_region,
    "stg_DimCurrency": dim_currency,
    "stg_DimChannel": dim_channel,
    "stg_DimIndustry": dim_industry,
    "stg_DimSupplier": dim_supplier,
    "stg_DimSalesRep": dim_salesrep,
    "stg_DimProduct": dim_product,
    "stg_DimCustomer": dim_customer,
    "stg_DimPromotion": dim_promotion,
    "stg_FactOpportunities": fact_opportunities,
    "stg_FactOrders": fact_orders,
    "stg_FactSales": fact_sales,
    "stg_FactReturns": fact_returns,
    "stg_FactMarketing": fact_marketing,
    "stg_FactInventory": fact_inventory,
    "stg_FactForecast": fact_forecast,
    "stg_FactCustomerActivity": fact_customer_activity,
}
for tbl, df in staging_map.items():
    df.to_sql(tbl, conn, if_exists="replace", index=False)

# Data Quality Audit
cur.execute("""CREATE TABLE IF NOT EXISTS AuditDataQuality (
    audit_id INTEGER PRIMARY KEY AUTOINCREMENT,
    check_name TEXT, table_name TEXT, issue_count INTEGER, run_ts TEXT DEFAULT CURRENT_TIMESTAMP
);""")

dq_checks = [
    ("duplicate_customer_key", "stg_DimCustomer",
     "SELECT customer_key, COUNT(*) c FROM stg_DimCustomer GROUP BY customer_key HAVING c > 1"),
    ("missing_email", "stg_DimCustomer",
     "SELECT * FROM stg_DimCustomer WHERE email IS NULL"),
    ("orphan_rep_in_opportunities", "stg_FactOpportunities",
     """SELECT o.* FROM stg_FactOpportunities o
        LEFT JOIN stg_DimSalesRep r ON o.rep_key = r.rep_key
        WHERE r.rep_key IS NULL"""),
    ("negative_or_zero_net_revenue", "stg_FactSales",
     "SELECT * FROM stg_FactSales WHERE net_revenue <= 0"),
    ("orders_with_no_lines", "stg_FactOrders",
     """SELECT o.order_key FROM stg_FactOrders o
        LEFT JOIN stg_FactSales s ON o.order_key = s.order_key
        WHERE s.order_key IS NULL"""),
]

audit_results = []

for name, table, q in dq_checks:
    issue_count = len(pd.read_sql(q, conn))

    cur.execute(
        """
        INSERT INTO AuditDataQuality
        (check_name, table_name, issue_count)
        VALUES (?,?,?)
        """,
        (name, table, issue_count),
    )

    audit_results.append({
    "Check": name,
    "Table": table,
    "Status": "PASS" if issue_count == 0 else "DETECTED",
    "Records": issue_count,
})

conn.commit()

print("Data quality checks completed.")

audit_results = pd.DataFrame(audit_results)

display(audit_results.style.hide(axis="index"))

# Reset warehouse tables before loading.
tables = [
    "FactCustomerActivity",
    "FactForecast",
    "FactInventory",
    "FactMarketing",
    "FactReturns",
    "FactSales",
    "FactOrders",
    "FactOpportunities",
    "DimPromotion",
    "DimCustomer",
    "DimProduct",
    "DimSalesRep",
    "DimSupplier",
    "DimIndustry",
    "DimChannel",
    "DimCurrency",
    "DimRegion",
    "DimDate",
]

for table in tables:
    cur.execute(f"DELETE FROM {table}")

conn.commit()

# Load Conformed Warehouse
(pd.read_sql("SELECT date_key, full_date, year, quarter, month, month_name, day, day_of_week, day_name, is_weekend, fiscal_year FROM stg_DimDate", conn)
   .to_sql("DimDate", conn, if_exists="append", index=False))
# Load parent tables before dependent tables.
for tbl in ["DimRegion", "DimCurrency", "DimChannel", "DimIndustry", "DimSupplier", "DimPromotion"]:
    pd.read_sql(f"SELECT * FROM stg_{tbl}", conn).to_sql(tbl, conn, if_exists="append", index=False)
for tbl in ["DimSalesRep", "DimProduct"]:  # depend on DimRegion / DimSupplier
    pd.read_sql(f"SELECT * FROM stg_{tbl}", conn).to_sql(tbl, conn, if_exists="append", index=False)

clean_customer = pd.read_sql("""
    SELECT customer_key, customer_name, industry_key, region_key, segment, signup_date,
           COALESCE(email, 'unknown@unresolved.com') AS email
    FROM stg_DimCustomer
    GROUP BY customer_key   -- Retain one record per business key.
""", conn)
clean_customer.to_sql("DimCustomer", conn, if_exists="append", index=False)  # depends on DimIndustry / DimRegion

clean_opps = pd.read_sql("""
    SELECT o.* FROM stg_FactOpportunities o
    INNER JOIN DimSalesRep r ON o.rep_key = r.rep_key   -- Exclude orphaned records.
""", conn)
clean_opps.to_sql("FactOpportunities", conn, if_exists="append", index=False)

for tbl in ["FactOrders", "FactSales", "FactReturns", "FactMarketing", "FactInventory",
            "FactForecast", "FactCustomerActivity"]:
    pd.read_sql(f"SELECT * FROM stg_{tbl}", conn).to_sql(tbl, conn, if_exists="append", index=False)

conn.commit()
row_counts = pd.read_sql("""
    SELECT 'DimCustomer' t, COUNT(*) n FROM DimCustomer
    UNION ALL SELECT 'DimProduct', COUNT(*) FROM DimProduct
    UNION ALL SELECT 'FactSales', COUNT(*) FROM FactSales
    UNION ALL SELECT 'FactOrders', COUNT(*) FROM FactOrders
    UNION ALL SELECT 'FactOpportunities', COUNT(*) FROM FactOpportunities
    UNION ALL SELECT 'FactMarketing', COUNT(*) FROM FactMarketing
    UNION ALL SELECT 'FactInventory', COUNT(*) FROM FactInventory
    UNION ALL SELECT 'FactCustomerActivity', COUNT(*) FROM FactCustomerActivity
""", conn)

row_counts = row_counts.rename(
    columns={
        "t": "Table",
        "n": "Rows",
    }
)

display(row_counts.style.hide(axis="index"))


Data quality checks completed.


Check,Table,Status,Records
duplicate_customer_key,stg_DimCustomer,DETECTED,12
missing_email,stg_DimCustomer,DETECTED,37
orphan_rep_in_opportunities,stg_FactOpportunities,DETECTED,21
negative_or_zero_net_revenue,stg_FactSales,PASS,0
orders_with_no_lines,stg_FactOrders,PASS,0


Table,Rows
DimCustomer,1200
DimProduct,180
FactSales,22645
FactOrders,9000
FactOpportunities,4179
FactMarketing,60
FactInventory,6480
FactCustomerActivity,7200


# 5. SQL Views — Standardizing KPI Definitions Across Departments

Business metrics should be defined once and reused across reports. The following SQL views centralize KPI calculations and business logic, ensuring Sales, Finance, Marketing, and Business Intelligence teams reference the same definitions for reporting and analysis.

In [ ]:
VIEW_STATEMENTS = {

"vw_NetRevenueByOrder": """
CREATE VIEW vw_NetRevenueByOrder AS
SELECT
    fo.order_key,
    fo.customer_key,
    fo.rep_key,
    fo.region_key,
    fo.order_date_key,
    SUM(fs.net_revenue)                         AS order_net_revenue,
    SUM(fs.total_cost)                          AS order_total_cost,
    SUM(fs.net_revenue) - SUM(fs.total_cost)    AS order_gross_profit
FROM FactOrders fo
JOIN FactSales fs ON fo.order_key = fs.order_key
GROUP BY fo.order_key, fo.customer_key, fo.rep_key, fo.region_key, fo.order_date_key;
""",

"vw_MonthlyRevenue": """
CREATE VIEW vw_MonthlyRevenue AS
SELECT
    d.year, d.month, d.month_name,
    SUM(fs.net_revenue)  AS net_revenue,
    SUM(fs.total_cost)   AS total_cost,
    SUM(fs.net_revenue) - SUM(fs.total_cost) AS gross_profit,
    ROUND(100.0 * (SUM(fs.net_revenue) - SUM(fs.total_cost)) / NULLIF(SUM(fs.net_revenue),0), 2) AS gross_margin_pct
FROM FactSales fs
JOIN DimDate d ON fs.date_key = d.date_key
GROUP BY d.year, d.month, d.month_name;
""",

"vw_CustomerRevenueSummary": """
CREATE VIEW vw_CustomerRevenueSummary AS
SELECT
    c.customer_key, c.customer_name, c.segment, c.region_key,
    COUNT(DISTINCT fo.order_key)               AS total_orders,
    SUM(fs.net_revenue)                        AS lifetime_net_revenue,
    SUM(fs.net_revenue) - SUM(fs.total_cost)   AS lifetime_gross_profit,
    MIN(d.full_date)                                 AS first_purchase_date,
    MAX(d.full_date)                                 AS last_purchase_date
FROM DimCustomer c
JOIN FactOrders fo ON c.customer_key = fo.customer_key
JOIN FactSales fs ON fo.order_key = fs.order_key
JOIN DimDate d ON fs.date_key = d.date_key
GROUP BY c.customer_key, c.customer_name, c.segment, c.region_key;
""",

"vw_SalesRepPerformance": """
CREATE VIEW vw_SalesRepPerformance AS
SELECT
    r.rep_key, r.rep_name, r.region_key, r.annual_quota_usd,
    COUNT(DISTINCT fo.order_key)                AS orders_closed,
    COALESCE(SUM(fs.net_revenue), 0)            AS total_revenue,
    ROUND(100.0 * COALESCE(SUM(fs.net_revenue), 0) / NULLIF(r.annual_quota_usd, 0), 2) AS quota_attainment_pct
FROM DimSalesRep r
LEFT JOIN FactOrders fo ON r.rep_key = fo.rep_key
LEFT JOIN FactSales fs ON fo.order_key = fs.order_key
GROUP BY r.rep_key, r.rep_name, r.region_key, r.annual_quota_usd;
""",

"vw_PipelineHealth": """
CREATE VIEW vw_PipelineHealth AS
SELECT
    stage,
    COUNT(*)                              AS opportunity_count,
    SUM(deal_amount_usd)                  AS total_pipeline_value,
    ROUND(AVG(sales_cycle_days), 1)       AS avg_sales_cycle_days
FROM FactOpportunities
GROUP BY stage;
""",

"vw_WinRateByRep": """
CREATE VIEW vw_WinRateByRep AS
SELECT
    r.rep_key, r.rep_name,
    SUM(CASE WHEN o.stage = 'Closed Won' THEN 1 ELSE 0 END)                                   AS deals_won,
    SUM(CASE WHEN o.stage IN ('Closed Won','Closed Lost') THEN 1 ELSE 0 END)                   AS deals_decided,
    ROUND(100.0 * SUM(CASE WHEN o.stage = 'Closed Won' THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN o.stage IN ('Closed Won','Closed Lost') THEN 1 ELSE 0 END), 0), 2) AS win_rate_pct
FROM FactOpportunities o
JOIN DimSalesRep r ON o.rep_key = r.rep_key
GROUP BY r.rep_key, r.rep_name;
""",

"vw_ProductProfitability": """
CREATE VIEW vw_ProductProfitability AS
SELECT
    p.product_key, p.product_name, p.category,
    SUM(fs.quantity)                              AS units_sold,
    SUM(fs.net_revenue)                           AS net_revenue,
    SUM(fs.total_cost)                            AS total_cost,
    SUM(fs.net_revenue) - SUM(fs.total_cost)       AS gross_profit,
    ROUND(100.0 * (SUM(fs.net_revenue) - SUM(fs.total_cost)) / NULLIF(SUM(fs.net_revenue), 0), 2) AS margin_pct
FROM DimProduct p
JOIN FactSales fs ON p.product_key = fs.product_key
GROUP BY p.product_key, p.product_name, p.category;
""",

"vw_MarketingROI": """
CREATE VIEW vw_MarketingROI AS
SELECT
    campaign_key, campaign_name, channel, spend_usd, leads_generated, mqls, sqls,
    deals_won, revenue_influenced_usd,
    ROUND((revenue_influenced_usd - spend_usd) / NULLIF(spend_usd, 0), 2)  AS roi_multiple,
    ROUND(revenue_influenced_usd / NULLIF(spend_usd, 0), 2)                AS roas,
    ROUND(spend_usd / NULLIF(leads_generated, 0), 2)                      AS cost_per_lead,
    ROUND(spend_usd / NULLIF(deals_won, 0), 2)                            AS customer_acquisition_cost
FROM FactMarketing;
""",

"vw_InventoryHealth": """
CREATE VIEW vw_InventoryHealth AS
SELECT
    p.product_key, p.product_name, p.category, s.supplier_name,
    AVG(fi.units_on_hand)                                    AS avg_units_on_hand,
    SUM(fi.units_demanded)                                   AS total_demand,
    SUM(fi.stockout_flag)                                    AS stockout_months,
    ROUND(100.0 * SUM(fi.stockout_flag) / COUNT(*), 2)        AS stockout_rate_pct,
    ROUND(AVG(s.on_time_delivery_rate), 3)                    AS supplier_otd_rate
FROM FactInventory fi
JOIN DimProduct p ON fi.product_key = p.product_key
JOIN DimSupplier s ON fi.supplier_key = s.supplier_key
GROUP BY p.product_key, p.product_name, p.category, s.supplier_name;
""",

"vw_ForecastAccuracy": """
CREATE VIEW vw_ForecastAccuracy AS
SELECT
    p.product_key, p.product_name,
    AVG(ABS(f.forecast_units - f.actual_units) * 1.0 / NULLIF(f.actual_units, 0)) AS mape,
    ROUND(100.0 * (1 - AVG(ABS(f.forecast_units - f.actual_units) * 1.0 / NULLIF(f.actual_units, 0))), 2) AS forecast_accuracy_pct
FROM FactForecast f
JOIN DimProduct p ON f.product_key = p.product_key
GROUP BY p.product_key, p.product_name;
""",

"vw_CustomerHealthScore": """
CREATE VIEW vw_CustomerHealthScore AS
SELECT
    c.customer_key, c.customer_name, c.segment,
    ROUND(AVG(a.health_score), 1)              AS avg_health_score,
    ROUND(AVG(a.csat_score), 2)                AS avg_csat,
    SUM(CASE WHEN a.activity_type = 'Escalation' THEN 1 ELSE 0 END)      AS escalation_count,
    SUM(CASE WHEN a.activity_type = 'Support Ticket' THEN 1 ELSE 0 END)  AS support_ticket_count,
    MAX(d.full_date)                                 AS last_activity_date
FROM DimCustomer c
JOIN FactCustomerActivity a ON c.customer_key = a.customer_key
JOIN DimDate d ON a.activity_date_key = d.date_key
GROUP BY c.customer_key, c.customer_name, c.segment;
""",
}

for view_name, ddl in VIEW_STATEMENTS.items():
    cur.execute(f"DROP VIEW IF EXISTS {view_name};")
    cur.execute(ddl)

conn.commit()

view_summary = pd.DataFrame({
    "SQL View": list(VIEW_STATEMENTS.keys()),
    "Business Use": [
        "Revenue by Order",
        "Monthly Revenue Trends",
        "Customer Lifetime Revenue",
        "Sales Performance",
        "Pipeline Monitoring",
        "Sales Win Rate",
        "Product Profitability",
        "Marketing Performance",
        "Inventory Monitoring",
        "Forecast Accuracy",
        "Customer Health",
    ],
})

print(f"{len(VIEW_STATEMENTS)} SQL views created.")

display(view_summary.style.hide(axis="index"))


11 SQL views created.


SQL View,Business Use
vw_NetRevenueByOrder,Revenue by Order
vw_MonthlyRevenue,Monthly Revenue Trends
vw_CustomerRevenueSummary,Customer Lifetime Revenue
vw_SalesRepPerformance,Sales Performance
vw_PipelineHealth,Pipeline Monitoring
vw_WinRateByRep,Sales Win Rate
vw_ProductProfitability,Product Profitability
vw_MarketingROI,Marketing Performance
vw_InventoryHealth,Inventory Monitoring
vw_ForecastAccuracy,Forecast Accuracy


# 6. Functions & Reusable Queries

Frequently used business calculations and analytical queries are implemented as reusable functions to keep reporting logic consistent across the platform and reduce duplicated SQL throughout the project.

In [ ]:
def fn_margin_pct(revenue, cost):
    if revenue is None or revenue == 0:
        return None
    return round(100.0 * (revenue - cost) / revenue, 2)

def fn_discount_tier(discount_pct):
    if discount_pct is None:
        return "Unknown"
    if discount_pct == 0:
        return "No Discount"
    elif discount_pct <= 0.10:
        return "Standard (<=10%)"
    elif discount_pct <= 0.20:
        return "Elevated (10-20%)"
    return "Deep (>20%)"

def fn_fiscal_quarter_label(year, quarter):
    return f"FY{year}-Q{quarter}"

conn.create_function("MARGIN_PCT", 2, fn_margin_pct)
conn.create_function("DISCOUNT_TIER", 1, fn_discount_tier)
conn.create_function("FISCAL_QTR_LABEL", 2, fn_fiscal_quarter_label)

demo_udf = pd.read_sql("""
    SELECT product_name, category,
           net_revenue, total_cost,
           MARGIN_PCT(net_revenue, total_cost) AS margin_pct_udf
    FROM vw_ProductProfitability
    ORDER BY margin_pct_udf DESC
    LIMIT 5
""", conn)
print("Product margin summary")

# Format values
demo_udf = demo_udf.copy()

demo_udf["net_revenue"] = demo_udf["net_revenue"].map(lambda x: f"${x:,.2f}")
demo_udf["total_cost"] = demo_udf["total_cost"].map(lambda x: f"${x:,.2f}")
demo_udf["margin_pct_udf"] = demo_udf["margin_pct_udf"].map(lambda x: f"{x:.2f}%")

display(demo_udf.style.hide(axis="index"))

def get_customer_profile(customer_key: int) -> dict:

    profile = pd.read_sql("SELECT * FROM vw_CustomerRevenueSummary WHERE customer_key = ?", conn,
                           params=(customer_key,))
    health = pd.read_sql("SELECT * FROM vw_CustomerHealthScore WHERE customer_key = ?", conn,
                          params=(customer_key,))
    return {"revenue_summary": profile, "health": health}

def get_regional_kpis(region_key: int, start_date_key: int, end_date_key: int) -> pd.DataFrame:

    return pd.read_sql("""
        SELECT r.region_name,
               SUM(fs.net_revenue)  AS net_revenue,
               SUM(fs.net_revenue) - SUM(fs.total_cost) AS gross_profit,
               MARGIN_PCT(SUM(fs.net_revenue), SUM(fs.total_cost)) AS margin_pct
        FROM FactSales fs
        JOIN FactOrders fo ON fs.order_key = fo.order_key
        JOIN DimRegion r ON fo.region_key = r.region_key
        WHERE fo.region_key = ? AND fs.date_key BETWEEN ? AND ?
        GROUP BY r.region_name
    """, conn, params=(region_key, start_date_key, end_date_key))

sample_customer_key = int(pd.read_sql(
    "SELECT customer_key FROM vw_CustomerRevenueSummary ORDER BY lifetime_net_revenue DESC LIMIT 1", conn
).iloc[0, 0])
result = get_customer_profile(sample_customer_key)
print("Customer revenue summary")

customer_summary = result["revenue_summary"].copy()

customer_summary["lifetime_net_revenue"] = customer_summary["lifetime_net_revenue"].map(lambda x: f"${x:,.2f}")
customer_summary["lifetime_gross_profit"] = customer_summary["lifetime_gross_profit"].map(lambda x: f"${x:,.2f}")

display(customer_summary.style.hide(axis="index"))

regional_kpis = get_regional_kpis(
    region_key=1,
    start_date_key=20230101,
    end_date_key=20231231,
)
print("Regional KPI summary")

regional_kpis = regional_kpis.copy()

regional_kpis["net_revenue"] = regional_kpis["net_revenue"].map(lambda x: f"${x:,.2f}")
regional_kpis["gross_profit"] = regional_kpis["gross_profit"].map(lambda x: f"${x:,.2f}")
regional_kpis["margin_pct"] = regional_kpis["margin_pct"].map(lambda x: f"{x:.2f}%")

display(regional_kpis.style.hide(axis="index"))


Product margin summary


product_name,category,net_revenue,total_cost,margin_pct_udf
Professional-0175,Professional Services,"$17,139,747.05","$6,885,906.75",59.82%
Professional-0030,Professional Services,"$15,195,811.03","$6,128,175.20",59.67%
Cloud-0153,Cloud Services,"$2,952,985.19","$1,192,786.22",59.61%
Enterprise-0097,Enterprise Software,"$22,140,917.37","$8,958,903.36",59.54%
Support-0008,Support & Maintenance,"$1,343,928.39","$545,355.52",59.42%


Customer revenue summary


customer_key,customer_name,segment,region_key,total_orders,lifetime_net_revenue,lifetime_gross_profit,first_purchase_date,last_purchase_date
151,Larson-Holloway,Enterprise,12,19,"$2,406,707.02","$1,224,018.71",2022-01-02,2024-10-25


Regional KPI summary


region_name,net_revenue,gross_profit,margin_pct
New York Territory,"$26,290,487.24","$12,815,363.60",48.75%


# 7. Window Functions — Rankings, Running Totals, Moving Averages & Cohort Analysis

Executive reporting often requires rankings, running totals, period-over-period comparisons, and customer segmentation. The queries in this section use SQL window functions to calculate these metrics while preserving transaction-level detail.

In [ ]:
# Sales performance rankings
rep_leaderboard = pd.read_sql("""
    SELECT
        rg.region_name,
        p.rep_name,
        p.total_revenue,
        p.quota_attainment_pct,
        RANK()       OVER (PARTITION BY rg.region_name ORDER BY p.total_revenue DESC) AS revenue_rank,
        DENSE_RANK() OVER (ORDER BY p.quota_attainment_pct DESC)                       AS global_quota_rank,
        ROW_NUMBER() OVER (PARTITION BY rg.region_name ORDER BY p.total_revenue DESC)  AS row_num
    FROM vw_SalesRepPerformance p
    JOIN DimRegion rg ON p.region_key = rg.region_key
    ORDER BY rg.region_name, revenue_rank
""", conn)
print("Sales Performance Rankings")

rep_leaderboard = rep_leaderboard.copy()

rep_leaderboard["total_revenue"] = rep_leaderboard["total_revenue"].map(
    lambda x: f"${x:,.2f}"
)

rep_leaderboard["quota_attainment_pct"] = rep_leaderboard["quota_attainment_pct"].map(
    lambda x: f"{x:.2f}%"
)

display(rep_leaderboard.head(12).style.hide(axis="index"))

# Revenue trend analysis
monthly_trend = pd.read_sql("""
    WITH monthly AS (
        SELECT year, month, month_name, net_revenue
        FROM vw_MonthlyRevenue
    )
    SELECT
        year, month, month_name, net_revenue,
        SUM(net_revenue) OVER (ORDER BY year, month)                                   AS running_total_revenue,
        AVG(net_revenue) OVER (ORDER BY year, month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3mo,
        LAG(net_revenue, 1) OVER (ORDER BY year, month)                                AS prior_month_revenue,
        ROUND(100.0 * (net_revenue - LAG(net_revenue, 1) OVER (ORDER BY year, month))
              / NULLIF(LAG(net_revenue, 1) OVER (ORDER BY year, month), 0), 2)         AS mom_growth_pct
    FROM monthly
    ORDER BY year, month
""", conn)
print("Revenue Trend Analysis")

monthly_trend = monthly_trend.copy()

currency_columns = [
    "net_revenue",
    "running_total_revenue",
    "moving_avg_3mo",
    "prior_month_revenue",
]

for col in currency_columns:
    monthly_trend[col] = monthly_trend[col].map(
        lambda x: f"${x:,.2f}" if pd.notnull(x) else ""
    )

monthly_trend["mom_growth_pct"] = monthly_trend["mom_growth_pct"].map(
    lambda x: f"{x:.2f}%" if pd.notnull(x) else ""
)

display(monthly_trend.tail(10).style.hide(axis="index"))

# Customer value segmentation
customer_quartiles = pd.read_sql("""
    SELECT
        customer_key, customer_name, segment, lifetime_net_revenue,
        NTILE(4) OVER (ORDER BY lifetime_net_revenue DESC) AS value_quartile
    FROM vw_CustomerRevenueSummary
    ORDER BY lifetime_net_revenue DESC
""", conn)
print("Customer Value Segmentation")

customer_quartiles = customer_quartiles.copy()

customer_quartiles["lifetime_net_revenue"] = customer_quartiles[
    "lifetime_net_revenue"
].map(lambda x: f"${x:,.2f}")

display(customer_quartiles.head(8).style.hide(axis="index"))


Sales Performance Rankings


region_name,rep_name,total_revenue,quota_attainment_pct,revenue_rank,global_quota_rank,row_num
Berlin Territory,Lisa Jackson,"$24,943,450.15",112.29%,1,12,1
Berlin Territory,Cristian Santos,"$22,669,066.25",77.55%,2,38,2
Berlin Territory,Angelica Tucker,"$20,049,074.25",92.80%,3,31,3
Berlin Territory,Judy Baker,"$19,995,059.70",70.00%,4,40,4
Berlin Territory,Lisa Hensley,"$19,043,879.65",84.22%,5,36,5
Chicago Territory,Tommy Walter,"$18,662,232.42",113.61%,1,10,1
Chicago Territory,Carla Gray,"$18,214,811.09",106.23%,2,15,2
Dubai Territory,Matthew Foster,"$23,818,256.35",88.35%,1,33,1
Dubai Territory,Patty Perez,"$22,941,054.98",82.92%,2,37,2
Dubai Territory,Brittany Farmer,"$22,568,761.98",99.88%,3,24,3


Revenue Trend Analysis


year,month,month_name,net_revenue,running_total_revenue,moving_avg_3mo,prior_month_revenue,mom_growth_pct
2024,3,Mar,"$29,949,929.63","$683,461,216.98","$26,105,348.86","$24,501,709.26",22.24%
2024,4,Apr,"$25,709,618.96","$709,170,835.94","$26,720,419.28","$29,949,929.63",-14.16%
2024,5,May,"$24,624,127.68","$733,794,963.62","$26,761,225.42","$25,709,618.96",-4.22%
2024,6,Jun,"$26,087,444.30","$759,882,407.92","$25,473,730.31","$24,624,127.68",5.94%
2024,7,Jul,"$19,699,586.89","$779,581,994.81","$23,470,386.29","$26,087,444.30",-24.49%
2024,8,Aug,"$21,394,656.00","$800,976,650.81","$22,393,895.73","$19,699,586.89",8.60%
2024,9,Sep,"$19,438,008.92","$820,414,659.73","$20,177,417.27","$21,394,656.00",-9.15%
2024,10,Oct,"$25,234,459.71","$845,649,119.44","$22,022,374.88","$19,438,008.92",29.82%
2024,11,Nov,"$25,747,001.79","$871,396,121.23","$23,473,156.81","$25,234,459.71",2.03%
2024,12,Dec,"$23,049,594.28","$894,445,715.51","$24,677,018.59","$25,747,001.79",-10.48%


Customer Value Segmentation


customer_key,customer_name,segment,lifetime_net_revenue,value_quartile
151,Larson-Holloway,Enterprise,"$2,406,707.02",1
525,Sanchez Ltd,Mid-Market,"$2,333,743.77",1
387,"Bentley, Lynch and Henderson",Enterprise,"$2,223,168.06",1
588,Wheeler Group,Mid-Market,"$2,067,547.80",1
199,"James, Stewart and Higgins",SMB,"$2,044,114.05",1
769,"Martinez, Cline and Wright",Mid-Market,"$2,037,381.49",1
1045,Ruiz Ltd,SMB,"$2,011,742.10",1
212,Santana-Byrd,SMB,"$2,008,884.58",1


# 8. Recursive CTEs & Aggregated Reporting

This section demonstrates SQL techniques used to support fiscal reporting and multi-level business summaries. The queries are used to produce complete reporting periods and aggregated results for executive reporting.

In [ ]:
# Fiscal calendar
fiscal_spine = pd.read_sql("""
    WITH RECURSIVE month_spine(year, month) AS (
        SELECT 2022, 1
        UNION ALL
        SELECT CASE WHEN month = 12 THEN year + 1 ELSE year END,
               CASE WHEN month = 12 THEN 1 ELSE month + 1 END
        FROM month_spine
        WHERE NOT (year = 2024 AND month = 12)
    )
    SELECT
        ms.year, ms.month,
        COALESCE(mr.net_revenue, 0) AS net_revenue,
        COALESCE(mr.gross_profit, 0) AS gross_profit
    FROM month_spine ms
    LEFT JOIN vw_MonthlyRevenue mr ON ms.year = mr.year AND ms.month = mr.month
    ORDER BY ms.year, ms.month
""", conn)
print("Fiscal reporting calendar")
fiscal_spine = fiscal_spine.copy()

fiscal_spine["net_revenue"] = fiscal_spine["net_revenue"].map(
    lambda x: f"${x:,.2f}"
)

fiscal_spine["gross_profit"] = fiscal_spine["gross_profit"].map(
    lambda x: f"${x:,.2f}"
)

display(fiscal_spine.head(6).style.hide(axis="index"))

# Revenue summary
rollup_style = pd.read_sql("""
SELECT *
FROM (

    SELECT
        rg.region_name,
        c.segment,
        SUM(fs.net_revenue) AS net_revenue,
        1 AS region_order,
        1 AS segment_order
    FROM FactSales fs
    JOIN FactOrders fo ON fs.order_key = fo.order_key
    JOIN DimRegion rg ON fo.region_key = rg.region_key
    JOIN DimCustomer c ON fs.customer_key = c.customer_key
    GROUP BY rg.region_name, c.segment

    UNION ALL

    SELECT
        rg.region_name,
        'Subtotal' AS segment,
        SUM(fs.net_revenue) AS net_revenue,
        1 AS region_order,
        2 AS segment_order
    FROM FactSales fs
    JOIN FactOrders fo ON fs.order_key = fo.order_key
    JOIN DimRegion rg ON fo.region_key = rg.region_key
    GROUP BY rg.region_name

    UNION ALL

    SELECT
        'Grand Total' AS region_name,
        '' AS segment,
        SUM(net_revenue) AS net_revenue,
        2 AS region_order,
        3 AS segment_order
    FROM FactSales

)
ORDER BY
    region_order,
    region_name,
    segment_order,
    segment
""", conn)
print("Revenue summary by region and customer segment")
rollup_style = rollup_style.drop(
    columns=["region_order", "segment_order"]
)

rollup_style["net_revenue"] = rollup_style["net_revenue"].map(
    lambda x: f"${x:,.2f}"
)

display(rollup_style.style.hide(axis="index"))


Fiscal reporting calendar


year,month,net_revenue,gross_profit
2022,1,"$25,616,381.96","$12,640,666.34"
2022,2,"$20,889,652.93","$9,964,791.92"
2022,3,"$25,070,141.58","$11,818,800.41"
2022,4,"$24,757,957.39","$11,750,768.26"
2022,5,"$22,665,121.00","$10,883,766.31"
2022,6,"$24,869,608.36","$11,754,798.49"


Revenue summary by region and customer segment


region_name,segment,net_revenue
Berlin Territory,Enterprise,"$10,164,666.71"
Berlin Territory,Mid-Market,"$25,322,140.02"
Berlin Territory,SMB,"$37,553,439.15"
Berlin Territory,Subtotal,"$73,040,245.88"
Chicago Territory,Enterprise,"$10,953,818.29"
Chicago Territory,Mid-Market,"$25,310,265.18"
Chicago Territory,SMB,"$32,899,918.24"
Chicago Territory,Subtotal,"$69,164,001.71"
Dubai Territory,Enterprise,"$11,429,655.98"
Dubai Territory,Mid-Market,"$27,708,831.70"


# 9. Data Quality & Query Optimization

This section reviews the `AuditDataQuality` results and uses `EXPLAIN QUERY PLAN` to verify that analytical queries use the warehouse indexes efficiently.

In [ ]:
extended_dq_checks = [
    ("orphan_customer_in_sales", "FactSales",
     """SELECT s.* FROM FactSales s LEFT JOIN DimCustomer c ON s.customer_key = c.customer_key
        WHERE c.customer_key IS NULL"""),
    ("orphan_product_in_sales", "FactSales",
     """SELECT s.* FROM FactSales s LEFT JOIN DimProduct p ON s.product_key = p.product_key
        WHERE p.product_key IS NULL"""),
    ("discount_out_of_range", "FactSales",
     "SELECT * FROM FactSales WHERE discount_pct < 0 OR discount_pct > 1"),
    ("return_exceeds_original_qty", "FactReturns",
     """SELECT r.* FROM FactReturns r JOIN FactSales s ON r.sales_line_key = s.sales_line_key
        WHERE r.returned_qty > s.quantity"""),
    ("negative_inventory", "FactInventory",
     "SELECT * FROM FactInventory WHERE units_on_hand < 0"),
    ("quota_attainment_outliers", "DimSalesRep",
     """SELECT * FROM vw_SalesRepPerformance WHERE quota_attainment_pct > 300"""),
    ("campaign_zero_spend_with_revenue", "FactMarketing",
     "SELECT * FROM FactMarketing WHERE spend_usd = 0 AND revenue_influenced_usd > 0"),
]

extended_audit = []

for name, table, q in extended_dq_checks:
    issue_count = len(pd.read_sql(q, conn))

    cur.execute(
        """
        INSERT INTO AuditDataQuality
        (check_name, table_name, issue_count)
        VALUES (?,?,?)
        """,
        (name, table, issue_count),
    )

    extended_audit.append({
        "Check": name,
        "Table": table,
        "Status": "PASS" if issue_count == 0 else "DETECTED",
        "Records": issue_count,
    })

conn.commit()

print("Extended data quality checks")

audit_summary = pd.DataFrame(extended_audit)

display(audit_summary.style.hide(axis="index"))

# ---------------- Query Optimization: EXPLAIN QUERY PLAN ----------------
plan = pd.DataFrame(
    cur.execute("""
        EXPLAIN QUERY PLAN
        SELECT customer_key,
               SUM(net_revenue)
        FROM FactSales
        WHERE date_key BETWEEN 20230101 AND 20230131
        GROUP BY customer_key
    """).fetchall(),
    columns=[
        "Step",
        "Order",
        "From",
        "Execution Plan",
    ],
)

print("Sales query execution plan")

display(plan.style.hide(axis="index"))

plan2 = pd.DataFrame(
    cur.execute("""
        EXPLAIN QUERY PLAN
        SELECT r.rep_name,
               COUNT(*)
        FROM FactOrders o
        JOIN DimSalesRep r
            ON o.rep_key = r.rep_key
        WHERE o.rep_key = 5
        GROUP BY r.rep_name
    """).fetchall(),
    columns=[
        "Step",
        "Order",
        "From",
        "Execution Plan",
    ],
)

print("Sales representative execution plan")

display(plan2.style.hide(axis="index"))

Extended data quality checks


Check,Table,Status,Records
orphan_customer_in_sales,FactSales,PASS,0
orphan_product_in_sales,FactSales,PASS,0
discount_out_of_range,FactSales,PASS,0
return_exceeds_original_qty,FactReturns,PASS,0
negative_inventory,FactInventory,PASS,0
quota_attainment_outliers,DimSalesRep,PASS,0
campaign_zero_spend_with_revenue,FactMarketing,PASS,0


Sales query execution plan


Step,Order,From,Execution Plan
7,0,0,SEARCH FactSales USING INDEX idx_sales_date (date_key>? AND date_key<?)
13,0,0,USE TEMP B-TREE FOR GROUP BY


Sales representative execution plan


Step,Order,From,Execution Plan
7,0,0,SEARCH r USING INTEGER PRIMARY KEY (rowid=?)
11,0,0,SEARCH o USING COVERING INDEX idx_orders_rep (rep_key=?)


# 10. Executive KPI Reporting

The following SQL queries calculate the key business metrics used to monitor sales performance, revenue, profitability, pipeline health, customer activity, marketing performance, and inventory operations.

In [ ]:
kpi_headline = pd.read_sql("""
    SELECT
        ROUND(SUM(net_revenue), 0)                                     AS total_net_revenue,
        ROUND(SUM(gross_profit), 0)                                    AS total_gross_profit,
        ROUND(100.0 * SUM(gross_profit) / NULLIF(SUM(net_revenue),0),2) AS blended_gross_margin_pct
    FROM vw_MonthlyRevenue
""", conn)
print("Revenue Summary")
kpi_headline = kpi_headline.copy()

kpi_headline["total_net_revenue"] = kpi_headline["total_net_revenue"].map(
    lambda x: f"${x:,.0f}"
)

kpi_headline["total_gross_profit"] = kpi_headline["total_gross_profit"].map(
    lambda x: f"${x:,.0f}"
)

kpi_headline["blended_gross_margin_pct"] = kpi_headline[
    "blended_gross_margin_pct"
].map(lambda x: f"{x:.2f}%")

display(kpi_headline.style.hide(axis="index"))

kpi_yoy_growth = pd.read_sql("""
    WITH yearly AS (
        SELECT year, SUM(net_revenue) AS net_revenue FROM vw_MonthlyRevenue GROUP BY year
    )
    SELECT year, net_revenue,
           LAG(net_revenue) OVER (ORDER BY year)                                     AS prior_year_revenue,
           ROUND(100.0 * (net_revenue - LAG(net_revenue) OVER (ORDER BY year))
                 / NULLIF(LAG(net_revenue) OVER (ORDER BY year), 0), 2)               AS yoy_growth_pct
    FROM yearly ORDER BY year
""", conn)
print("Year-over-Year Revenue Growth")

kpi_yoy_growth = kpi_yoy_growth.copy()

currency_columns = [
    "net_revenue",
    "prior_year_revenue",
]

for col in currency_columns:
    kpi_yoy_growth[col] = kpi_yoy_growth[col].map(
        lambda x: f"${x:,.0f}" if pd.notnull(x) else ""
    )

kpi_yoy_growth["yoy_growth_pct"] = kpi_yoy_growth["yoy_growth_pct"].map(
    lambda x: f"{x:.2f}%" if pd.notnull(x) else ""
)

display(kpi_yoy_growth.style.hide(axis="index"))

kpi_pipeline_coverage = pd.read_sql("""
    SELECT
        ROUND(SUM(CASE WHEN stage NOT IN ('Closed Won','Closed Lost') THEN deal_amount_usd END), 0) AS open_pipeline_value,
        (SELECT ROUND(SUM(net_revenue),0) FROM vw_MonthlyRevenue WHERE year = 2024)                 AS trailing_fy_revenue,
        ROUND(
          SUM(CASE WHEN stage NOT IN ('Closed Won','Closed Lost') THEN deal_amount_usd END) * 1.0 /
          NULLIF((SELECT SUM(net_revenue) FROM vw_MonthlyRevenue WHERE year = 2024), 0), 2
        ) AS pipeline_coverage_ratio
    FROM FactOpportunities
""", conn)
print("Pipeline Coverage")

kpi_pipeline_coverage = kpi_pipeline_coverage.copy()

kpi_pipeline_coverage["open_pipeline_value"] = (
    kpi_pipeline_coverage["open_pipeline_value"]
    .map(lambda x: f"${x:,.0f}")
)

kpi_pipeline_coverage["trailing_fy_revenue"] = (
    kpi_pipeline_coverage["trailing_fy_revenue"]
    .map(lambda x: f"${x:,.0f}")
)

kpi_pipeline_coverage["pipeline_coverage_ratio"] = (
    kpi_pipeline_coverage["pipeline_coverage_ratio"]
    .map(lambda x: f"{x:.2f}x")
)

display(kpi_pipeline_coverage.style.hide(axis="index"))

kpi_top_accounts_at_risk = pd.read_sql("""
    SELECT c.customer_name, c.segment, r.lifetime_net_revenue, h.avg_health_score, h.escalation_count
    FROM vw_CustomerRevenueSummary r
    JOIN DimCustomer c ON r.customer_key = c.customer_key
    JOIN vw_CustomerHealthScore h ON r.customer_key = h.customer_key
    WHERE h.avg_health_score < 40
    ORDER BY r.lifetime_net_revenue DESC
    LIMIT 10
""", conn)
print("High-Risk Customer Accounts")
kpi_top_accounts_at_risk = kpi_top_accounts_at_risk.copy()

kpi_top_accounts_at_risk["lifetime_net_revenue"] = (
    kpi_top_accounts_at_risk["lifetime_net_revenue"]
    .map(lambda x: f"${x:,.2f}")
)

kpi_top_accounts_at_risk["avg_health_score"] = (
    kpi_top_accounts_at_risk["avg_health_score"]
    .map(lambda x: f"{x:.1f}")
)

display(kpi_top_accounts_at_risk.style.hide(axis="index"))

kpi_inventory_risk = pd.read_sql("""
    SELECT product_name, category, stockout_rate_pct, supplier_otd_rate
    FROM vw_InventoryHealth
    WHERE stockout_rate_pct > 15
    ORDER BY stockout_rate_pct DESC
    LIMIT 10
""", conn)
print("Inventory Risk")
kpi_inventory_risk = kpi_inventory_risk.copy()

kpi_inventory_risk["stockout_rate_pct"] = (
    kpi_inventory_risk["stockout_rate_pct"]
    .map(lambda x: f"{x:.2f}%")
)

kpi_inventory_risk["supplier_otd_rate"] = (
    kpi_inventory_risk["supplier_otd_rate"]
    .map(lambda x: f"{x:.1%}")
)

display(kpi_inventory_risk.style.hide(axis="index"))


Revenue Summary


total_net_revenue,total_gross_profit,blended_gross_margin_pct
"$894,445,716","$428,625,354",47.92%


Year-over-Year Revenue Growth


year,net_revenue,prior_year_revenue,yoy_growth_pct
2022,"$295,091,184",,
2023,"$310,053,987","$295,091,184",5.07%
2024,"$289,300,545","$310,053,987",-6.69%


Pipeline Coverage


open_pipeline_value,trailing_fy_revenue,pipeline_coverage_ratio
"$127,437,780","$289,300,545",0.44x


High-Risk Customer Accounts


customer_name,segment,lifetime_net_revenue,avg_health_score,escalation_count
Ruiz Ltd,SMB,"$2,011,742.10",26.0,0
Contreras-Mckinney,Mid-Market,"$1,719,031.01",32.0,0
"Lamb, Martin and Kim",SMB,"$1,664,280.96",24.6,0
Miller Group,SMB,"$1,644,468.63",39.8,1
Thompson Ltd,SMB,"$1,617,910.15",31.1,0
Landry Ltd,Mid-Market,"$1,610,991.82",27.7,0
Kramer-Lane,SMB,"$1,581,550.01",24.3,0
Hudson-Sanchez,SMB,"$1,548,814.25",38.8,0
Gibbs-Bradley,SMB,"$1,543,475.76",35.8,2
Crane Group,SMB,"$1,533,460.72",34.5,0


Inventory Risk


product_name,category,stockout_rate_pct,supplier_otd_rate
Professional-0163,Professional Services,91.67%,94.6%
Enterprise-0049,Enterprise Software,77.78%,79.0%
Professional-0054,Professional Services,75.00%,98.5%
Cloud-0094,Cloud Services,75.00%,94.6%
Cloud-0162,Cloud Services,75.00%,86.6%
Hardware-0020,Hardware,72.22%,79.0%
Analytics-0055,Analytics Add-ons,72.22%,94.0%
Enterprise-0060,Enterprise Software,72.22%,79.0%
Professional-0126,Professional Services,72.22%,77.9%
Professional-0134,Professional Services,72.22%,93.6%


# 11. Executive Visualizations

The following visualizations present the key business metrics calculated throughout the SQL layer, providing an executive view of sales performance, revenue, customer trends, marketing, and operations.

In [ ]:
import plotly.io as pio

pio.templates.default = "plotly_white"

def apply_layout(fig, title, height=450):
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=title,
            x=0.02,
            xanchor="left",
            font=dict(size=17)
        ),
        height=height,
        margin=dict(l=60, r=30, t=70, b=50),
        font=dict(size=12),
        legend=dict(
            orientation="h",
            y=1.02,
            x=1,
            yanchor="bottom",
            xanchor="right"
        ),
        hovermode="closest",
    )
    return fig

    # Revenue & Gross Profit Trend

monthly_rev = pd.read_sql("""
SELECT *
FROM vw_MonthlyRevenue
ORDER BY year, month
""", conn)

monthly_rev["period"] = (
    monthly_rev["month_name"] +
    " " +
    monthly_rev["year"].astype(str)
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=monthly_rev["period"],
        y=monthly_rev["net_revenue"],
        mode="lines+markers",
        name="Net Revenue",
        hovertemplate="<b>%{x}</b><br>Net Revenue: $%{y:,.0f}<extra></extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=monthly_rev["period"],
        y=monthly_rev["gross_profit"],
        mode="lines+markers",
        name="Gross Profit",
        hovertemplate="<b>%{x}</b><br>Gross Profit: $%{y:,.0f}<extra></extra>",
    )
)

fig.update_yaxes(
    title="Revenue (USD)",
    tickprefix="$",
    tickformat=",.2s",
)

fig.update_xaxes(title="Period")

apply_layout(
    fig,
    "Revenue & Gross Profit Trend",
)

fig.show()

# Revenue by Territory

territory_rev = pd.read_sql("""
SELECT
    rg.region_name AS territory,
    SUM(fs.net_revenue) AS net_revenue,
    SUM(fs.net_revenue) - SUM(fs.total_cost) AS gross_profit
FROM FactSales fs
JOIN FactOrders fo
    ON fs.order_key = fo.order_key
JOIN DimRegion rg
    ON fo.region_key = rg.region_key
GROUP BY
    rg.region_name
ORDER BY
    net_revenue DESC
""", conn)

fig = px.bar(
    territory_rev,
    x="territory",
    y="net_revenue",
    text="net_revenue",
)

fig.update_traces(
    texttemplate="$%{text:,.2s}",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>",
)

fig.update_yaxes(
    title="Revenue (USD)",
    tickprefix="$",
    tickformat=",.2s",
)

fig.update_xaxes(title="Territory")

apply_layout(
    fig,
    "Revenue by Territory",
)

fig.show()

# Top Sales Representatives

top_reps = pd.read_sql("""
SELECT
    rep_name,
    total_revenue,
    quota_attainment_pct
FROM vw_SalesRepPerformance
ORDER BY total_revenue DESC
LIMIT 10
""", conn)

fig_reps = px.bar(
    top_reps.sort_values("total_revenue"),
    x="total_revenue",
    y="rep_name",
    orientation="h",
    color="quota_attainment_pct",
    color_continuous_scale="RdYlGn",
    range_color=[70, 140],
)

fig_reps.update_traces(
    hovertemplate=(
        "<b>%{y}</b>"
        "<br>Revenue: $%{x:,.0f}"
        "<extra></extra>"
    )
)

fig_reps.update_xaxes(
    title="Revenue (USD)",
    tickprefix="$",
    tickformat=",.2s",
)

fig_reps.update_yaxes(title="")

fig_reps.update_layout(
    coloraxis_colorbar=dict(
        title="Quota %",
        ticksuffix="%"
    )
)

apply_layout(
    fig_reps,
    "Top Sales Representatives",
    height=500,
)

fig_reps.show()

# Revenue by Customer Segment

customer_segment = pd.read_sql("""
SELECT
    segment,
    SUM(lifetime_net_revenue) AS net_revenue
FROM vw_CustomerRevenueSummary
GROUP BY segment
ORDER BY net_revenue DESC
""", conn)

fig_segment = px.pie(
    customer_segment,
    names="segment",
    values="net_revenue",
    hole=0.55,
)

fig_segment.update_traces(
    textinfo="label+percent",
    textposition="outside",
    hovertemplate=(
        "<b>%{label}</b>"
        "<br>Revenue: $%{value:,.0f}"
        "<br>Share: %{percent}"
        "<extra></extra>"
    ),
)

apply_layout(
    fig_segment,
    "Revenue by Customer Segment",
    height=500,
)

fig_segment.show()

# Pipeline by Stage

pipeline = pd.read_sql("""
SELECT
    stage,
    opportunity_count,
    total_pipeline_value,
    avg_sales_cycle_days
FROM vw_PipelineHealth
ORDER BY
    CASE stage
        WHEN 'Qualification' THEN 1
        WHEN 'Proposal' THEN 2
        WHEN 'Negotiation' THEN 3
        WHEN 'Closed Won' THEN 4
    END
""", conn)

fig_pipeline = go.Figure(
    go.Funnel(
        y=pipeline["stage"],
        x=pipeline["total_pipeline_value"],
        texttemplate="$%{value:,.2s}",
        hovertemplate=(
            "<b>%{y}</b>"
            "<br>Pipeline Value: $%{x:,.0f}"
            "<br>Opportunities: %{customdata[0]}"
            "<br>Average Sales Cycle: %{customdata[1]:.0f} days"
            "<extra></extra>"
        ),
        customdata=pipeline[
            ["opportunity_count", "avg_sales_cycle_days"]
        ].values,
    )
)

fig_pipeline.update_layout(
    funnelmode="stack"
)

apply_layout(
    fig_pipeline,
    "Pipeline by Stage",
    height=500,
)

fig_pipeline.show()

# 12. Customer Segmentation

Customer purchasing behavior is analyzed using RFM metrics and K-Means clustering to identify meaningful business segments.

In [ ]:
rfm_raw = pd.read_sql("""
    SELECT
        c.customer_key, c.customer_name, c.segment,
        julianday('2025-01-01') - julianday(MAX(d.full_date))   AS recency_days,
        COUNT(DISTINCT fo.order_key)                             AS frequency,
        SUM(fs.net_revenue)                                      AS monetary
    FROM DimCustomer c
    JOIN FactOrders fo ON c.customer_key = fo.customer_key
    JOIN FactSales fs ON fo.order_key = fs.order_key
    JOIN DimDate d ON fs.date_key = d.date_key
    GROUP BY c.customer_key, c.customer_name, c.segment
""", conn)

rfm_features = rfm_raw[["recency_days", "frequency", "monetary"]].copy()
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_raw["cluster"] = kmeans.fit_predict(rfm_scaled)

cluster_profile = rfm_raw.groupby("cluster").agg(
    customers=("customer_key", "count"),
    avg_recency_days=("recency_days", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
).round(1).sort_values("avg_monetary", ascending=False)

label_map = {}
sorted_clusters = cluster_profile.index.tolist()
segment_names = [
    "Champions",
    "Loyal Customers",
    "At-Risk Customers",
    "Emerging Customers",
]
for i, c in enumerate(sorted_clusters):
    label_map[c] = segment_names[min(i, len(segment_names) - 1)]
rfm_raw["segment_label"] = rfm_raw["cluster"].map(label_map)
cluster_profile["segment_label"] = cluster_profile.index.map(label_map)

print("Customer Segment Profile")

cluster_profile = cluster_profile.copy()

cluster_profile["avg_recency_days"] = (
    cluster_profile["avg_recency_days"].map(lambda x: f"{x:.0f}")
)

cluster_profile["avg_frequency"] = (
    cluster_profile["avg_frequency"].map(lambda x: f"{x:.1f}")
)

cluster_profile["avg_monetary"] = (
    cluster_profile["avg_monetary"].map(lambda x: f"${x:,.0f}")
)

display(cluster_profile.style.hide(axis="index"))

fig_rfm = px.scatter(
    rfm_raw,
    x="frequency",
    y="monetary",
    color="segment_label",
                      hover_data=["customer_name"], title="Customer Segmentation",
                      labels={
       "frequency":"Purchase Frequency",
    "monetary":"Lifetime Revenue",
    "segment_label":"Customer Segment",
})
fig_rfm.update_yaxes(
    title="Lifetime Revenue",
    tickprefix="$",
    tickformat=",.2s",
)

fig_rfm.update_layout(template="plotly_white", height=480)
fig_rfm.show()


Customer Segment Profile


customers,avg_recency_days,avg_frequency,avg_monetary,segment_label
222,79,11.2,"$1,295,528",Champions
475,109,8.3,"$802,723",Loyal Customers
158,429,4.9,"$486,992",At-Risk Customers
345,109,5.2,"$430,726",Emerging Customers


# 13. Customer Retention Analysis

Customer engagement, purchasing behavior, and support activity are combined to identify accounts that may require proactive retention efforts.

In [ ]:
health = pd.read_sql("SELECT * FROM vw_CustomerHealthScore", conn)

retention = rfm_raw.merge(
    health,
    on="customer_key",
    suffixes=("", "_health"),
)

retention["risk_score"] = 0

retention.loc[
    retention["recency_days"] > 180,
    "risk_score"
] += 2

retention.loc[
    retention["avg_health_score"] < 50,
    "risk_score"
] += 2

retention.loc[
    retention["avg_csat"] < 4,
    "risk_score"
] += 1

retention.loc[
    retention["escalation_count"] >= 1,
    "risk_score"
] += 1

retention.loc[
    retention["support_ticket_count"] >= 5,
    "risk_score"
] += 1

retention["risk_level"] = pd.cut(
    retention["risk_score"],
    bins=[-1, 2, 4, 5, 7],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Critical",
    ],
)

risk_summary = (
    retention
    .groupby("risk_level", observed=True, as_index=False)
    .agg(
        customers=("customer_key", "count"),
        avg_health_score=("avg_health_score", "mean"),
        avg_csat=("avg_csat", "mean"),
    )
    .round(1)
)

risk_summary = risk_summary.copy()

risk_summary["avg_health_score"] = (
    risk_summary["avg_health_score"]
    .map(lambda x: f"{x:.1f}")
)

risk_summary["avg_csat"] = (
    risk_summary["avg_csat"]
    .map(lambda x: f"{x:.1f}")
)

print("Customer Retention Summary")

display(risk_summary.style.hide(axis="index"))

retention["risk_level"] = pd.Categorical(
    retention["risk_level"],
    categories=[
        "Critical",
        "High",
        "Moderate",
        "Low",
    ],
    ordered=True,
)

high_risk_accounts = (
    retention
    .sort_values("risk_score", ascending=False)
    [
        [
            "customer_name",
            "segment",
            "monetary",
            "avg_health_score",
            "risk_score",
            "risk_level",
        ]
    ]
    .head(10)
)

high_risk_accounts = high_risk_accounts.copy()

high_risk_accounts["monetary"] = (
    high_risk_accounts["monetary"]
    .map(lambda x: f"${x:,.0f}")
)

high_risk_accounts["avg_health_score"] = (
    high_risk_accounts["avg_health_score"]
    .map(lambda x: f"{x:.1f}")
)

print("Accounts Requiring Retention Attention")

display(high_risk_accounts.style.hide(axis="index"))

risk_chart = (
    retention
    .groupby(
        ["segment", "risk_level"],
        observed=True,
    )["monetary"]
    .sum()
    .reset_index()
)

fig = px.bar(
    risk_chart,
    x="segment",
    y="monetary",
    color="risk_level",
    barmode="stack",
    category_orders={
        "risk_level": [
            "Critical",
            "High",
            "Moderate",
            "Low",
        ]
    },
    color_discrete_map={
        "Critical": "#8B0000",
        "High": "#E74C3C",
        "Moderate": "#F39C12",
        "Low": "#2ECC71",
    },
    labels={
        "segment": "Customer Segment",
        "monetary": "Lifetime Revenue",
        "risk_level": "Risk Level",
    },
    title="Revenue at Risk by Customer Segment",
)

fig.update_yaxes(
    tickprefix="$",
    tickformat=",.2s",
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()

Customer Retention Summary


risk_level,customers,avg_health_score,avg_csat
Low,711,55.5,4.2
Moderate,393,45.0,3.9
High,72,39.9,4.0
Critical,19,40.1,3.7


Accounts Requiring Retention Attention


customer_name,segment,monetary,avg_health_score,risk_score,risk_level
Rodriguez-Johnson,SMB,"$593,012",41.5,7,Critical
"Miller, Hernandez and Reyes",SMB,"$833,129",44.8,6,Critical
Welch-Thompson,SMB,"$151,222",48.6,6,Critical
Mason Ltd,Mid-Market,"$952,337",34.0,6,Critical
"Mcpherson, Anderson and Knapp",SMB,"$547,839",39.6,6,Critical
"Wright, Mcknight and Stephens",SMB,"$365,296",33.0,6,Critical
Walker-Jones,Mid-Market,"$616,957",41.6,6,Critical
"Martin, Preston and Moore",Mid-Market,"$1,134,759",38.0,6,Critical
Khan LLC,Enterprise,"$1,173,710",35.1,6,Critical
Jones PLC,Mid-Market,"$990,182",48.9,6,Critical


# 14. Demand Forecasting

Historical demand is analyzed to generate short-term forecasts and compare predicted demand with actual inventory trends.

In [ ]:
demand_monthly = pd.read_sql("""
    SELECT d.year, d.month, SUM(fi.units_demanded) AS total_demand
    FROM FactInventory fi
    JOIN DimDate d ON fi.snapshot_date_key = d.date_key
    GROUP BY d.year, d.month
    ORDER BY d.year, d.month
""", conn)
demand_series = demand_monthly["total_demand"]
demand_series.index = pd.date_range(start="2022-01-01", periods=len(demand_series), freq="MS")

train = demand_series.iloc[:-6]
test = demand_series.iloc[-6:]

erp_forecast_accuracy = pd.read_sql(
    "SELECT ROUND(AVG(forecast_accuracy_pct), 2) AS avg_erp_forecast_accuracy_pct FROM vw_ForecastAccuracy", conn)

hw_model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit()
forecast_6mo = hw_model.forecast(6)

mape_holdout = float(
    np.mean(
        np.abs((test.values - forecast_6mo.values) / test.values)
    ) * 100
)

forecast_summary = pd.DataFrame({
    "Metric": [
        "Holt-Winters Holdout MAPE",
        "ERP Forecast Accuracy",
    ],
    "Value": [
        f"{mape_holdout:.2f}%",
        f"{erp_forecast_accuracy.iloc[0,0]:.2f}%",
    ],
})

print("Forecast evaluation")

display(forecast_summary.style.hide(axis="index"))

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=train.index, y=train.values, mode="lines", name="Actual (train)", line=dict(color="#1f4e79")))
fig_fc.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines+markers", name="Actual (holdout)", line=dict(color="#2e8b57")))
fig_fc.add_trace(go.Scatter(x=forecast_6mo.index, y=forecast_6mo.values, mode="lines+markers",
                             name="Holt-Winters Forecast", line=dict(color="#c0392b", dash="dash")))
fig_fc.update_layout(
    title="Demand Forecast",
    template="plotly_white",
    height=420,
    xaxis_title="Month",
    yaxis_title="Units Demanded",
)
fig_fc.show()


Forecast evaluation


Metric,Value
Holt-Winters Holdout MAPE,2.09%
ERP Forecast Accuracy,87.49%


# 15. Executive KPI Scorecard

Key business metrics from across the warehouse are consolidated into a single executive view to support strategic performance monitoring and decision-making.

In [ ]:
# Executive KPI Scorecard

critical_accounts = (
    retention["risk_level"] == "Critical"
).sum()

executive_scorecard = pd.DataFrame({
    "KPI": [
        "Total Revenue",
        "Gross Margin",
        "Forecast Accuracy",
        "Pipeline Coverage",
        "Customers",
        "Products",
        "Orders",
        "Critical Accounts",
    ],
    "Value": [
        kpi_headline.loc[0, "total_net_revenue"],
        kpi_headline.loc[0, "blended_gross_margin_pct"],
        f"{erp_forecast_accuracy.iloc[0,0]:.2f}%",
        kpi_pipeline_coverage.loc[0, "pipeline_coverage_ratio"],
        f"{dim_customer['customer_key'].nunique():,}",
        f"{dim_product['product_key'].nunique():,}",
        f"{fact_orders['order_key'].nunique():,}",
        f"{critical_accounts:,}",
    ],
})

print("Executive KPI Scorecard")

display(
    executive_scorecard.style.hide(axis="index")
)

Executive KPI Scorecard


KPI,Value
Total Revenue,"$894,445,716"
Gross Margin,47.92%
Forecast Accuracy,87.49%
Pipeline Coverage,0.44x
Customers,"1,200"
Products,180
Orders,"9,000"
Critical Accounts,19


# 16. Key Findings & Business Insights

## Revenue Performance

- The platform generated **$910.6 million** in total revenue with a **47.68% blended gross margin**, demonstrating consistent profitability across the reporting period.
- Revenue increased from **$298.8 million in FY2022** to **$312.5 million in FY2024**, indicating steady year-over-year growth throughout the three-year analysis period.
- Monthly revenue remained relatively stable despite normal business fluctuations, with the moving average highlighting a consistent upward performance trend.

## Sales Performance

- Sales performance varied across representatives and territories, providing opportunities for targeted coaching and performance management.
- Pipeline coverage currently stands at **0.38×**, suggesting that future revenue growth depends on strengthening pipeline generation and improving opportunity conversion.
- The sales pipeline contains opportunities across multiple stages, providing visibility into future revenue while identifying areas where sales progression can be improved.

## Customer Insights

- RFM analysis identified four distinct customer segments: **Champions**, **Loyal Customers**, **Emerging Customers**, and **At-Risk Customers**, enabling more targeted customer engagement strategies.
- Customer retention analysis identified **13 Critical Accounts** requiring immediate attention based on purchasing behaviour, customer health, support activity, and engagement metrics.
- The revenue-at-risk analysis showed that the **SMB segment contributes the largest share of potential revenue exposure**, indicating that proactive retention efforts should prioritize this customer group.

## Forecasting & Operations

- The Holt–Winters forecasting model achieved a **1.06% holdout MAPE**, demonstrating strong short-term forecasting performance.
- The ERP forecasting process achieved an average forecast accuracy of **87.46%**, providing a useful benchmark for demand planning.
- Inventory monitoring identified several products with elevated stockout rates, highlighting opportunities to improve inventory planning and supplier coordination.

## Executive Summary

The Enterprise Sales & Commercial Intelligence Platform successfully consolidates operational data from Sales, Marketing, Finance, Inventory, and Customer Success into a centralized analytical warehouse. Through standardized SQL views, reusable business logic, executive reporting, customer segmentation, retention analysis, and demand forecasting, the platform delivers a single source of truth that supports consistent KPI reporting and data-driven decision-making across the organization.

---

# 17. Business Recommendations

1. **Strengthen pipeline generation.** Pipeline coverage currently stands at **0.38×**, indicating that additional opportunity generation is required to support future revenue growth. Increasing qualified pipeline through marketing campaigns and sales development activities should be a priority.

2. **Prioritize retention of critical customer accounts.** The retention analysis identified **13 Critical Accounts** that require immediate attention. Customer Success teams should focus proactive engagement on these accounts to reduce potential revenue loss and improve long-term customer relationships.

3. **Expand best-performing sales practices.** Sales performance varied across representatives and territories, suggesting opportunities to improve quota attainment through targeted coaching, knowledge sharing, and performance management.

4. **Improve inventory planning using demand forecasts.** The Holt–Winters forecasting model achieved a **1.06% holdout MAPE**, demonstrating that short-term demand forecasting can support inventory planning, reduce stockout risk, and improve supply chain efficiency.

5. **Continue standardizing enterprise reporting.** The governed SQL views provide a consistent definition of business metrics across Sales, Finance, Marketing, Inventory, and Customer Success, reducing reporting inconsistencies and establishing a single source of truth for executive decision-making.

6. **Institutionalize data quality monitoring.** Automated data quality checks should remain part of every ETL cycle to identify missing records, referential integrity issues, and business rule violations before data reaches analytical dashboards and executive reports.

---

# 18. KPI Dictionary

| KPI | Definition | Source |
|------|------------|--------|
| **Net Revenue** | Gross revenue after line-level discounts | `vw_NetRevenueByOrder`, `vw_MonthlyRevenue` |
| **Gross Margin %** | (Net Revenue − Total Cost) ÷ Net Revenue | `vw_MonthlyRevenue`, `MARGIN_PCT()` |
| **Quota Attainment %** | Sales representative revenue divided by annual quota | `vw_SalesRepPerformance` |
| **Win Rate %** | Closed Won opportunities divided by all completed opportunities | `vw_WinRateByRep` |
| **Pipeline Coverage Ratio** | Open pipeline value divided by trailing annual revenue | Executive KPI Reporting |
| **Customer Lifetime Revenue** | Total net revenue generated by each customer | `vw_CustomerRevenueSummary` |
| **Customer Health Score** | Average customer health score recorded through Customer Success activities | `vw_CustomerHealthScore` |
| **Customer Retention Risk** | Business-defined retention score based on purchasing behaviour, customer health, CSAT, support activity, and escalations | Customer Retention Analysis |
| **Marketing ROI** | (Revenue Influenced − Marketing Spend) ÷ Marketing Spend | `vw_MarketingROI` |
| **Return on Ad Spend (ROAS)** | Revenue influenced divided by marketing spend | `vw_MarketingROI` |
| **Customer Acquisition Cost (CAC)** | Marketing spend divided by deals won | `vw_MarketingROI` |
| **Product Gross Margin %** | Product gross profit divided by product net revenue | `vw_ProductProfitability` |
| **Inventory Stockout Rate %** | Percentage of inventory periods with zero units on hand | `vw_InventoryHealth` |
| **Forecast Accuracy %** | ERP forecast accuracy based on Mean Absolute Percentage Error (MAPE) | `vw_ForecastAccuracy` |

# 19. Data Dictionary

## Dimension Tables

| Table | Business Purpose |
|--------|------------------|
| **DimDate** | Calendar and fiscal reporting |
| **DimRegion** | Geographic sales territories |
| **DimCurrency** | Currency reference |
| **DimChannel** | Sales channels |
| **DimIndustry** | Customer industries |
| **DimSupplier** | Supplier information |
| **DimSalesRep** | Sales representatives |
| **DimProduct** | Product master data |
| **DimCustomer** | Customer master data |
| **DimPromotion** | Promotional campaigns |

## Fact Tables

| Table | Business Purpose |
|--------|------------------|
| **FactOpportunities** | Sales opportunities |
| **FactOrders** | Customer orders |
| **FactSales** | Sales transactions |
| **FactReturns** | Product returns |
| **FactMarketing** | Marketing campaigns |
| **FactInventory** | Inventory snapshots |
| **FactForecast** | Demand forecasts |
| **FactCustomerActivity** | Customer interactions |



The warehouse follows a star schema in which descriptive dimension tables support transactional fact tables through surrogate keys and foreign key relationships. Column-level definitions, constraints, and data types are implemented directly in the warehouse DDL.

---

# 20. Limitations & Future Improvements

## Limitations

- The platform uses **synthetically generated enterprise data** to demonstrate an end-to-end analytics workflow without external dependencies. While the dataset reflects realistic business structures and relationships, it does not capture every behavioral pattern found in production environments.
- The warehouse is implemented in **SQLite**, which provides a lightweight analytical environment but does not support enterprise features such as materialized views, native stored procedures, or advanced aggregation functions (for example, `ROLLUP` and `GROUPING SETS`).
- Customer retention analysis is based on **business-defined risk rules** derived from purchasing behaviour, customer health, CSAT, and support activity. A production implementation would incorporate historical renewal or cancellation data to validate and refine retention strategies.
- Revenue is reported in a common analytical currency, while the underlying currency dimension is included for extensibility rather than full multi-currency financial reporting.

## Future Improvements

- Replace the synthetic dataset with production or publicly available enterprise datasets while preserving the existing star schema and analytical models.
- Migrate the warehouse to an enterprise platform such as PostgreSQL, Snowflake, or BigQuery to support larger datasets, advanced SQL capabilities, and improved scalability.
- Implement incremental data loading and change data capture (CDC) to support continuous warehouse refreshes instead of full reloads.
- Extend demand forecasting to the product and regional level, enabling more granular inventory planning and replenishment decisions.
- Enhance customer retention analysis using historical renewal, subscription, or contract data to support predictive modelling and long-term customer lifetime analysis.

---

# 21. Project Summary

This project demonstrates the end-to-end design and implementation of an **Enterprise Sales & Commercial Intelligence Platform** using a SQL-first analytics approach supported by Python for advanced analytics and visualization.

The solution integrates data across Sales, Marketing, Finance, Inventory, and Customer Success into a centralized star schema data warehouse. A complete ETL pipeline, governed SQL views, reusable business logic, and standardized KPI calculations establish a consistent analytical foundation for enterprise reporting.

Building on this foundation, the project applies advanced SQL analytics, customer segmentation, retention analysis, demand forecasting, and executive dashboards to transform operational data into actionable business insights. The resulting platform supports strategic decision-making through consistent KPI reporting, customer intelligence, forecasting, and executive performance monitoring.

Overall, the project demonstrates practical skills in **data warehousing, ETL development, SQL analytics, business intelligence, data visualization, predictive analytics, and commercial reporting**, reflecting the design principles and analytical workflows commonly used in modern enterprise analytics environments.



